In [ ]:
# Use as many python cells as you wish to write your code
import pandas as pd

def merge_all_data(health_file, supplement_file, experiments_file, profiles_file):
    health = pd.read_csv(health_file)
    supplement = pd.read_csv(supplement_file)
    experiments = pd.read_csv(experiments_file)
    profiles = pd.read_csv(profiles_file)

    # Fix data types
    health["date"] = pd.to_datetime(health["date"])
    supplement["date"] = pd.to_datetime(supplement["date"])

    supplement["dosage_grams"] = supplement["dosage"]

    # Clean supplement dosage (mg → grams)
    supplement.loc[supplement["dosage_unit"] == "mg", "dosage_grams"] = supplement["dosage"] / 1000

    # merge the supplement
    supplement = supplement.merge(
        experiments,
        on="experiment_id",
        how="left"
    )

    supplement = supplement.rename(columns={
        "name": "experiment_name"
    })

    # merge all table
    df = health.merge(profiles, on="user_id", how="left")
    
    df = df.merge(
        supplement,
        on=["user_id", "date"],
        how="left"
    )
    
    # Handling missing supplement data
    df["supplement_name"] = df["supplement_name"].fillna("No intake")

    # Create a group age
    def age_group(age):
        if pd.isna(age):
            return "Unknown"
        elif age < 18:
            return "Under 18"
        elif age <= 25:
            return "18-25"
        elif age <= 35:
            return "26-35"
        elif age <= 45:
            return "36-45"
        elif age <= 55:
            return "46-55"
        elif age <= 65:
            return "56-65"
        else:
            return "Over 65"

    df["user_age_group"] = df["age"].apply(age_group)

    # Select require column
    df = df[[
        "user_id",
        "date",
        "email",
        "user_age_group",
        "experiment_name",
        "supplement_name",
        "dosage_grams",
        "is_placebo",
        "average_heart_rate",
        "average_glucose",
        "sleep_hours",
        "activity_level"
    ]]

    return df